---
title: Sampler examples
description: Practical examples of using the Sampler primitive in Qiskit Runtime.
---


# Sampler examples

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.2
qiskit-ibm-runtime~=0.47.0
```
</AccordionItem>
</Accordion>

Generate entire error-mitigated quasi-probability distributions sampled from quantum circuit outputs. Leverage Sampler’s capabilities for search and classification algorithms like Grover’s and QVSM.

## Run a single experiment

Use Sampler to return the measurement outcome as bitstrings or counts of a single circuit.

In [1]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]

print(f" > First ten results: {pub_result.data.meas.get_bitstrings()[:10]}")

 > First ten results: ['0110010000101111000000100101111111001000000001001011100110000000100100000001010000001000011100001001010001001000000110011010000', '1000111011010101001001001000010100101110100000100000010110000010001000110010110100101010000010000000010001010000000010000001000', '0000010110010011011011101011101011011010110011100000111111011010100000101000000101000000000000010001010000010001000010010100001', '0011000011001010101101011001101001010010100001101010000001010000000100100000100111001001010110100000110000000010101000010000100', '0100000100001010001001001010011011111100110010000101000100001000000101101000000010000000000010110100101101101000110011000001010', '0100011001101001010100111111000110100001000000000001010011001000100101100010010100010010100110110000010010010010000010000000010', '1110011011101111001110001101110000101001110000001011100100000000111000011001110000000000001000000011100000001000101101001000100', '11111101100110000000011110000001010011001100000001111000110

## Run multiple experiments in a single job

Use Sampler to return the measurement outcome as bitstrings or counts of multiple circuits in one job.

In [2]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [iqp(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits)
result = job.result()

for idx, pub_result in enumerate(result):
    print(
        f" > First five results for pub {idx}: "
        f"{pub_result.data.meas.get_bitstrings()[:5]}"
    )

 > First five results for pub 0: ['1000011010110000001100110001000000110101000000010001000010100101110000011101010000001000001100100101000100000100000100100100111', '0001110011001111011011111000010101100010110000100011100001011110000001100000000111100100010110100010011101101101011001001010100', '1100001110001011010101000001110010000110010010011101000000001001100010011010000010001100000010000000010011100101000010100110011', '0101000001111000110011110110111100111101110001011010101111000100010000111001000000000011100000000010000001001100101000000010000', '0100010010100001101010101000011101110010000111011000010101000010100000100110110111000010011100000000100000100000000000100110000']
 > First five results for pub 1: ['0000001010100011110010010001000111011010000000101101101111001110111001101000010110010100110101010011110101100000001001000111000', '1101000000001011101000010001100000100000111111100000110110001001110101001000110101000000101110000100000000111010100001101110010', '10111110110001

## Run parameterized circuits

Run several experiments in a single job, leveraging parameter values to increase circuit reusability.

In [3]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

# Step 1: Map classical inputs to a quantum problem
circuit = real_amplitudes(num_qubits=n_qubits, reps=2)
circuit.measure_all()

# Define three sets of parameters for the circuit
rng = np.random.default_rng(1234)
parameter_values = [
    rng.uniform(-np.pi, np.pi, size=circuit.num_parameters) for _ in range(3)
]

# Step 2: Optimize problem for quantum execution.

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

# Step 3: Execute using Qiskit primitives.
sampler = Sampler(backend)
job = sampler.run([(isa_circuit, parameter_values)])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
# Get counts from the classical register "meas".
print(
    f" >> First five results for the meas output register: "
    f"{pub_result.data.meas.get_bitstrings()[:5]}"
)

 >> First five results for the meas output register: ['1011110101010110000010101001001110000101100111001011001011101100011110010010001010000100011111111100000111100000111010110000111', '0010101011101011001110100111101010100000100101111001010111101001110100011010001110101110111101010101111010111011001010010000101', '0000000111110000001000101101100001001111010011010011000011101111000001110000001110101111100101110000110101011001001111010000001', '0110110101011101010010001011100010010110101000001101111000000001010110101010011010010000010101110110111011110111111100010001001', '0000110011110111100100000000101110110111010001000010011000010010001101101111010110010001001000000110110010010111111010010000111']


## Use batches and advanced options

Explore the batch [execution mode](/docs/guides/execution-modes) and advanced options to optimize circuit performance on QPUs.

In [4]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Batch, SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng(1234)
mat = np.real(random_hermitian(n_qubits, seed=rng))
circuit = iqp(mat)
circuit.measure_all()
mat = np.real(random_hermitian(n_qubits, seed=rng))
another_circuit = iqp(mat)
another_circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
another_isa_circuit = pm.run(another_circuit)

# The context manager automatically closes the batch.
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    job = sampler.run([isa_circuit])
    another_job = sampler.run([another_isa_circuit])
    result = job.result()
    another_result = another_job.result()

# first job

print(
    f" > The first five measurement results of job 1: "
    f"{result[0].data.meas.get_bitstrings()[:5]}"
)

 > The first five measurement results of job 1: ['0100001101100001011001011010010110101010010000000110000010111001000101001100100100110000101001000101101001101000000010111100001', '1101101101000010101100011100000001101000000001000011100000011001001000000101100000000100100000000011100000000011001000001010100', '1110101010100110101011011101110100000011111110000001011011100100001011000010010110010111101010100111100010001010000000001000000', '1100111100100001010110111110000000001011000001000011111000100010010001100110110101000000001000011001101001001010011000001111001', '1011001110101111010111000111000111011010100100110000100110001000010100110101010001100010010000000101000001011000110000001000110']


In [5]:
# second job
print(
    " > The first five measurement results of job 2:",
    another_result[0].data.meas.get_bitstrings()[:5],
)

 > The first five measurement results of job 2: ['1001110100010110001100111000000011010010010011101011101101001010000011010010000100001100001000001100000000010000111000101011110', '1101111001010001011100011010011110010000100010011010110101111000000000100010100100111100001000000000000000000000100010001000000', '0001100000001100011100100000000010010100010000100010110000110010001101110110000001000100100010000000000000010000110000000010100', '1011010011100000101010001001010000110011101011101010101001111000101001011000000000000010001010010010010011000000001000000010100', '0111101100111010010001101010010000100010101011011100001000100010010111000010000100000010110000000001101000111000100000100011000']


## Next steps

<Admonition type="tip" title="Recommendations">
    - [Specify advanced runtime options](runtime-options-overview).
    - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
    - Learn how to transpile locally in the [Transpile](/docs/guides/transpile/) section.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
    - Understand the [Job limits](/docs/guides/job-limits) when sending a job to an IBM&reg; QPU.
</Admonition>